# Laboratorio: Evaluación No Supervisada (GMM, KDE y DBSCAN)
## Análisis de Patrones Asistenciales en Hospitalizaciones de Diabetes (1999–2008)
**Machine Learning II · Parcial 2**

Este cuaderno implementa de manera unificada y rigurosa las tres metodologías no supervisadas de la Semana 08:
1. **GMM (Gaussian Mixture Models)**: Estimación de densidad paramétrica, cálculo de responsabilidades y selección objetiva de componentes mediante BIC/AIC.
2. **KDE (Kernel Density Estimation)**: Estimación no paramétrica continua y selección de ancho de banda $h$ por validación cruzada.
3. **DBSCAN**: Detección de núcleos densos de atención hospitalaria y segregación de casos clínicos atípicos (ruido) previa estandarización z-score.

**Unidad de análisis**: Primer encuentro por paciente único (71.518 observaciones).
**Espacio de representación**: Días de estancia hospitalaria (`time_in_hospital`) vs. Intensidad farmacológica (`num_medications`).

In [ ]:
from pathlib import Path
import sys, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from threadpoolctl import threadpool_limits

# Configuración de rutas y utilidades de salud
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from salud import (load_clinical_data, setup, savefig, save_table, clinical_axes,
                   ellipses, safe_silhouette, density_grid, COLORS)
setup()
threadpool_limits(limits=2)

# Carga de datos con partición estricta de desarrollo
frame = load_clinical_data(kind="desarrollo")
X = frame[["time_in_hospital", "num_medications"]].to_numpy()
print(f"Desarrollo: {len(frame):,} pacientes únicos. Unidades: días y cantidad de fármacos. Semilla: 42.")
print("Test temporal/independiente completamente cerrado y congelado hasta evaluación final.")
display(frame[["encounter_id", "patient_nbr", "age", "time_in_hospital", "num_medications", "num_lab_procedures", "readmitted"]].head(5))

## 1. Auditoría y Justificación de Datos
La base de datos original contiene 101.766 atenciones hospitalarias. La unidad de análisis clínico relevante es el **paciente único**;
por ello, se filtra el primer encuentro hospitalario registrado (71.518 pacientes), eliminando sesgos de autocorrelación por reingresos frecuentes.
No se eliminan estancias prolongadas ni recetas altas porque representan complejidad asistencial real, no errores de digitación.
El espacio analítico bidimensional conserva unidades naturales interpretables: días y número de medicamentos administrados.

In [ ]:
audit_df = pd.read_csv(ROOT / "results/tables/auditoria_clinica.csv")
display(audit_df)

## 2. Contrato Experimental

| Dimensión Metodológica | Declaración y Justificación en Salud |
| :--- | :--- |
| **Unidad de análisis** | Paciente diabético hospitalizado en su primer encuentro registrado ($n=71.518$). |
| **Variables analíticas** | $X_1$: Estancia hospitalaria (días, rango 1-14); $X_2$: Fármacos administrados (rango 1-81). |
| **Protocolo de partición** | Partición estratificada 80% desarrollo / 20% test con semilla fija (`random_state=42`). |
| **Aislamiento de test** | El conjunto de test permanece strictly congelado durante la selección de hiperparámetros. |
| **Objetivo asistencial** | Detectar perfiles de carga hospitalaria, núcleos densos de tratamiento y casos atípicos extremos. |
| **Límites de inferencia** | No se asume causalidad diagnóstica sin variables de desenlace posterior.|

## 3. Modelo 1: Mezclas Gaussianas (GMM) y Selección por Complejidad
El modelo de mezclas gaussianas representa la densidad conjunta como una combinación convexa de normales multivariadas:
$$p(x) = \sum_{k=1}^K \pi_k \mathcal{N}(x; \mu_k, \Sigma_k), \quad \sum_{k=1}^K \pi_k = 1.$$
Las responsabilidades condicionales corresponden a:
$$\gamma_{ik} = \frac{\pi_k \mathcal{N}(x_i; \mu_k, \Sigma_k)}{\sum_{j=1}^K \pi_j \mathcal{N}(x_i; \mu_j, \Sigma_j)}.$$
Para la selección del número de componentes $K$ y la estructura de covarianza (`full` vs `diag`) usamos BIC y AIC:
$$\mathrm{BIC} = -2\ell(\widehat{\theta}) + p\log n, \quad \mathrm{AIC} = -2\ell(\widehat{\theta}) + 2p.$$

In [ ]:
from sklearn.mixture import GaussianMixture
gmm_results = pd.read_csv(ROOT / "results/tables/gmm_seleccion.csv")
display(gmm_results.head(10))

# Ajuste del modelo ganador
best_row = gmm_results.iloc[0]
k_opt = int(best_row["k"])
cov_opt = str(best_row["covariance"])
print(f"Modelo GMM Seleccionado: K={k_opt} con covarianza {cov_opt} (BIC = {best_row['bic']:.2f})")
gmm_best = GaussianMixture(n_components=k_opt, covariance_type=cov_opt, random_state=42, reg_covar=1e-3).fit(X)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for cov_type, marker in [("full", "o-"), ("diag", "s--")]:
    sub = gmm_results[gmm_results["covariance"] == cov_type].sort_values("k")
    axes[0].plot(sub["k"], sub["bic"], marker, label=f"BIC ({cov_type})")
    axes[0].plot(sub["k"], sub["aic"], marker, alpha=0.5, label=f"AIC ({cov_type})")
axes[0].set_title("Selección de Componentes GMM (BIC / AIC)", pad=14, fontweight="bold")
axes[0].set_xlabel("Número de componentes (K)")
axes[0].set_ylabel("Puntuación")
axes[0].legend()

labels_gmm = gmm_best.predict(X)
axes[1].scatter(X[:, 0], X[:, 1], c=[COLORS[i % len(COLORS)] for i in labels_gmm], s=6, alpha=0.3)
ellipses(axes[1], gmm_best)
clinical_axes(axes[1], X, f"GMM (K={k_opt}, {cov_opt}) - Elipses de Covarianza al 95%")
plt.tight_layout()
plt.show()

## 4. Modelo 2: Estimación de Densidad por Kernel (KDE)
KDE modela la función de densidad continua no paramétrica acumulando la masa local de cada observación:
$$\widehat{f}_h(x) = \frac{1}{n h^d} \sum_{i=1}^n K\left(\frac{x - x_i}{h}\right).$$
El compromiso sesgo-varianza se gobierna por el ancho de banda $h$: valores pequeños generan sobreajuste ruidoso;
valores altos diluyen patrones multimodales. Evaluamos $h$ mediante validación cruzada de 3 pliegues reportando la log-densidad media.

In [ ]:
from sklearn.neighbors import KernelDensity
kde_cv = pd.read_csv(ROOT / "results/tables/kde_seleccion.csv")
display(kde_cv)

h_opt = float(kde_cv.loc[kde_cv["mean_log_density"].idxmax(), "h"])
print(f"Ancho de banda óptimo por CV: h = {h_opt}")
kde_best = KernelDensity(bandwidth=h_opt, kernel="gaussian").fit(X)

# Superficie de densidad 2D
xx, yy, grid = density_grid(X)
dens = np.exp(kde_best.score_samples(grid)).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].errorbar(kde_cv["h"], kde_cv["mean_log_density"], yerr=kde_cv["std"], fmt="o-")
axes[0].set_title("Validación Cruzada de Ancho de Banda (KDE)", pad=14, fontweight="bold")
axes[0].set_xlabel("Ancho de banda h")
axes[0].set_ylabel("Log-densidad media")

cs = axes[1].contourf(xx, yy, dens, levels=14, cmap="Blues")
axes[1].scatter(X[:, 0], X[:, 1], color="#087f8c", s=3, alpha=0.15)
plt.colorbar(cs, ax=axes[1], label="Densidad continua estimada")
clinical_axes(axes[1], X, f"Superficie KDE (h={h_opt})")
plt.tight_layout()
plt.show()

## 5. Modelo 3: Agrupamiento Basado en Densidad (DBSCAN)
DBSCAN formaliza vecindades cerradas de radio $\varepsilon$ sobre los datos estandarizados (`StandardScaler`):
$$N_{\varepsilon}(x_i) = \{x_j : \|x_i - x_j\| \leq \varepsilon\}.$$
Un punto es **núcleo** si $|N_{\varepsilon}(x_i)| \ge m$, expandiendo el cluster a puntos densamente alcanzables.
Aquellas observaciones sin densidad suficiente quedan etiquetadas como **ruido (-1)**.
En datos clínicos de hospitalización, el ruido aísla **casos atípicos de alta complejidad y polifarmacia extrema**.
Calibramos $\varepsilon$ mediante la curva del $m$-ésimo vecino en datos estandarizados.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Estandarización z-score para equilibrar escalas entre días y fármacos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Curva de k-vecinos en datos estandarizados
fig, ax = plt.subplots(figsize=(7, 3.8))
for m in [5, 10, 15, 20, 30]:
    distances, _ = NearestNeighbors(n_neighbors=m).fit(X_scaled).kneighbors(X_scaled)
    kth = np.sort(distances[:, m-1])
    ax.plot(np.arange(len(kth))/len(kth), kth, label=f"m={m}")
ax.set_title("Calibración de Distancia al m-ésimo Vecino", pad=14, fontweight="bold")
ax.set_xlabel("Fracción acumulada de observaciones")
ax.set_ylabel("Distancia al m-ésimo vecino (std)")
ax.legend()
plt.tight_layout()
plt.show()

# Tabla de sensibilidad
db_sens = pd.read_csv(ROOT / "results/tables/dbscan_sensibilidad.csv")
display(db_sens.head(10))

# Configuración seleccionada sobre datos estandarizados
chosen_eps, chosen_m = 0.3, 15
db_final = DBSCAN(eps=chosen_eps, min_samples=chosen_m).fit(X_scaled)
labels_db = db_final.labels_
noise_pct = np.mean(labels_db == -1) * 100
print(f"DBSCAN seleccionado (eps={chosen_eps}, m={chosen_m}): {len(np.unique(labels_db[labels_db!=-1]))} clusters, {noise_pct:.2f}% de atípicos/ruido")

## 6. Comité de Movilidad y Complejidad Hospitalaria
Integramos las tres fuentes complementarias de evidencia sobre el mismo espacio de pacientes:
- **GMM**: Elipses de covarianza al 95% para componentes superpuestas.
- **KDE**: Curvas de nivel continuas del núcleo de mayor concentración hospitalaria.
- **DBSCAN**: Puntos atípicos segregados en baja densidad (ruido).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
# DBSCAN ajustado sobre datos estandarizados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
db_final = DBSCAN(eps=0.3, min_samples=15).fit(X_scaled)
labels_db = db_final.labels_
is_noise = (labels_db == -1)
axes[0].scatter(X[is_noise, 0], X[is_noise, 1], c="lightgrey", s=8, label="Ruido / Atípicos")
axes[0].scatter(X[~is_noise, 0], X[~is_noise, 1], c=[COLORS[i % len(COLORS)] for i in labels_db[~is_noise]], s=8, alpha=0.6, label="Clusters densos")
axes[0].legend()
clinical_axes(axes[0], X, "DBSCAN (eps=0.3, min_samples=15)")

axes[1].contour(xx, yy, dens, levels=6, cmap="Greys", alpha=0.7)
axes[1].scatter(X[is_noise, 0], X[is_noise, 1], c="red", s=6, alpha=0.35, label="Atípicos DBSCAN")
ellipses(axes[1], gmm_best)
axes[1].legend()
clinical_axes(axes[1], X, "Comité Tripartito: GMM, KDE y DBSCAN")
plt.tight_layout()
plt.show()

## 7. Evaluación en Conjunto de Test Congelado
Una vez seleccionados e hiperparametrizados los modelos sin contaminar la muestra de prueba,
se abre el conjunto de test independiente ($n=1.500$) para certificar la generalización probabilística.

In [ ]:
test_results = pd.read_csv(ROOT / "results/tables/evaluacion_test_congelado.csv")
display(test_results)

## 8. Recomendación y Defensa Metodológica

> **Observo**: Una concentración multimodal densa en estancias cortas (1 a 4 días) con 10 a 20 medicamentos, modelada con precisión por $K=6$ en GMM y $h=0,2$ en KDE; paralelamente, DBSCAN aísla un 3,75% de pacientes en regiones periféricas de larga estadía (>8 días) y polifarmacia (>30 fármacos).
>
> **Interpreto**: La atención hospitalaria de diabetes no conforma grupos esféricos discretos sino un gradiente continuo de severidad clínica con un núcleo masivo ambulatorio-agudo y colas pesadas de casos complejos.
>
> **Recomendaría estudiar**: Protocolos diferenciados para el núcleo de alta rotación (días 1–3, fármacos 8–18) para reducir tiempos innecesarios de cama, y un programa de farmacovigilancia especializada para el grupo intermedio (días 5–9, fármacos 20–35).
>
> **Vigilaría**: Las observaciones clasificadas como ruido en DBSCAN, correspondientes a pacientes con estancias mayores a 8 días y más de 25 fármacos, dado que representan el mayor riesgo de eventos adversos y costos hospitalarios.
>
> **No puedo concluir**: Que pertenecer a un componente gaussiano específico determine la causalidad de reingreso o la efectividad del tratamiento sin incorporar covariables de comorbilidad y pruebas diagnósticas longitudinales.